# Scenario: AsyncOgxClient SDK Coverage

Validates that `AsyncOgxClient` from `ogx_client` operates properly with async/await methods across key endpoints:
- **Inspect / Health:** `client.inspect.health()`
- **Models:** `client.models.list()`
- **Chat Completions:** `client.chat.completions.create()`
- **Embeddings:** `client.embeddings.create()`

## Setup & Initialization

Load configuration from environment variables and initialize `AsyncOgxClient`.

In [ ]:
import os
from ogx_client import AsyncOgxClient

base_url = os.environ.get("BASE_URL", "http://localhost:8321")
model = os.environ.get("INFERENCE_MODEL")
embedding_model = os.environ.get("EMBEDDING_MODEL")
embedding_dimension = int(os.environ.get("EMBEDDING_DIMENSION", "768"))

assert base_url, "BASE_URL must be set"
assert model, "INFERENCE_MODEL must be set"

ogx_base_url = base_url.rstrip("/")
ogx_base_url = ogx_base_url if ogx_base_url.endswith("/v1") else ogx_base_url + "/v1"

client = AsyncOgxClient(base_url=ogx_base_url)

## Server Health Check (`inspect.health`)

Verify `await client.inspect.health()` completes and returns valid health status.

In [ ]:
health = await client.inspect.health()
assert health is not None, "Expected health response to be non-None"
assert (
    getattr(health, "status", None) == "OK"
    or health == "OK"
    or hasattr(health, "status")
), f"Health check failed or unexpected response: {health!r}"

## Models List (`models.list`)

Verify `await client.models.list()` returns the list of registered models.

In [ ]:
models_resp = await client.models.list()
assert models_resp is not None, "Expected models response"
model_ids = []
if hasattr(models_resp, "data"):
    model_ids = [m.id for m in models_resp.data]
elif isinstance(models_resp, list):
    model_ids = [getattr(m, "id", str(m)) for m in models_resp]

assert len(model_ids) > 0, "Expected at least one model in list"
assert any(model in mid or mid in model for mid in model_ids), (
    f"Configured model {model!r} not found in model IDs: {model_ids}"
)

## Chat Completions (`chat.completions.create`)

Verify async chat completion with `await client.chat.completions.create()`.

In [ ]:
chat_resp = await client.chat.completions.create(
    model=model,
    messages=[{"role": "user", "content": "Reply with exactly one word: Hello"}],
    temperature=0.0,
)
assert chat_resp is not None, "Expected chat completion response"
assert hasattr(chat_resp, "choices") and len(chat_resp.choices) > 0, (
    "Expected non-empty choices"
)
content = chat_resp.choices[0].message.content
assert content and len(content.strip()) > 0, "Expected non-empty message content"

## Embeddings (`embeddings.create`)

Verify async embedding creation with `await client.embeddings.create()` if an embedding model is configured.

In [ ]:
if embedding_model:
    emb_resp = await client.embeddings.create(
        model=embedding_model,
        input="Async client embedding test",
    )
    assert emb_resp is not None, "Expected embedding response"
    assert hasattr(emb_resp, "data") and len(emb_resp.data) > 0, (
        "Expected embedding data"
    )
    vector = emb_resp.data[0].embedding
    assert len(vector) == embedding_dimension, (
        f"Expected vector dimension {embedding_dimension}, got {len(vector)}"
    )
else:
    print("EMBEDDING_MODEL not set, skipping async embedding test")
    assert True

## Context Manager & Cleanup

Verify that `AsyncOgxClient` works as an async context manager and closes properly.

In [ ]:
async with AsyncOgxClient(base_url=ogx_base_url) as async_client:
    h = await async_client.inspect.health()
    assert h is not None

await client.close()